In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

path = "/content/drive/MyDrive/CICIDS2017/WebAttacks-Thursday-no-metadata.parquet"
df = pd.read_parquet(path)

print(df.shape)
df.head()

(155820, 78)


,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,6,113095465,48,24,9668,10012,403,0,201.416672,203.548294,...,32,203985.500,5.758372e+05,1629110,379,13800000.0,4277541.0,16500000,6737603,Benign
1,6,113473706,68,40,11364,12718,403,0,167.117645,171.919418,...,32,178326.875,5.034269e+05,1424245,325,13800000.0,4229413.0,16500000,6945512,Benign
2,0,119945515,150,0,0,0,0,0,0.000000,0.000000,...,0,6909777.500,1.170000e+07,20400000,6,24400000.0,24300000.0,60100000,5702188,Benign
3,6,60261928,9,7,2330,4221,1093,0,258.888885,409.702148,...,20,0.000,0.000000e+00,0,0,0.0,0.0,0,0,Benign
4,17,269,2,2,102,322,51,51,51.000000,0.000000,...,32,0.000,0.000000e+00,0,0,0.0,0.0,0,0,Benign


In [ ]:
df['Label'].unique()

['Benign', 'Web Attack � Brute Force', 'Web Attack � XSS', 'Web Attack � Sql Injection']
Categories (4, object): ['Benign', 'Web Attack � Brute Force', 'Web Attack � Sql Injection',
                         'Web Attack � XSS']

In [ ]:
# Clean label column (important)
df['Label'] = df['Label'].astype(str).str.strip()

# Binary target for Stage-1 VAE
df['target'] = df['Label'].apply(
    lambda x: 0 if x.lower() == 'benign' else 1
)

# Verify
df[['Label', 'target']].value_counts()

,,count
Label,target,
Benign,0,153677
Web Attack � Brute Force,1,1470
Web Attack � XSS,1,652
Web Attack � Sql Injection,1,21


In [ ]:
df['target'].value_counts()

,count
target,
0,153677
1,2143


In [ ]:
df.groupby(['Label', 'target']).size()

,,0
Label,target,
Benign,0,153677
Web Attack � Brute Force,1,1470
Web Attack � Sql Injection,1,21
Web Attack � XSS,1,652


In [ ]:
X = df.drop(columns=['Label', 'target'])
y = df['target']


In [ ]:
X.columns

Index(['Protocol', 'Flow Duration', 'Total Fwd Packets',
       'Total Backward Packets', 'Fwd Packets Length Total',
       'Bwd Packets Length Total', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
       'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags',
       'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Packet Length Min', 'Packet Length Max', 'Packet Length Mean',
       'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count',
       'SYN Flag Count', 'RST Fla

In [ ]:
X = pd.get_dummies(X, columns=['Protocol'], drop_first=True)
X_normal = X[y == 0]   # Benign only
X_attack = X[y == 1]   # Attacks ( testing)

In [ ]:
import pandas as pd

dos_path = "/content/drive/MyDrive/CICIDS2017/DoS-Wednesday-no-metadata.parquet"
df_dos = pd.read_parquet(dos_path)

print(df_dos.shape)
df_dos.head()

(584991, 78)


,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,6,38308,1,1,6,6,6,6,6.000000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
1,6,479,11,5,172,326,79,0,15.636364,31.449238,...,32,0.0,0.0,0,0,0.0,0.0,0,0,Benign
2,6,1095,10,6,3150,3150,1575,0,315.000000,632.561646,...,32,0.0,0.0,0,0,0.0,0.0,0,0,Benign
3,6,15206,17,12,3452,6660,1313,0,203.058823,425.778473,...,32,0.0,0.0,0,0,0.0,0.0,0,0,Benign
4,6,1092,9,6,3150,3152,1575,0,350.000000,694.509705,...,32,0.0,0.0,0,0,0.0,0.0,0,0,Benign


In [ ]:
df_dos['target'] = df_dos['Label'].apply(
    lambda x: 0 if x.lower() == 'benign' else 1
)

threshold=0.809063018200581

In [ ]:
X_dos = df_dos.drop(columns=['Label', 'target'])

# Same encoding as before
X_dos = pd.get_dummies(X_dos, columns=['Protocol'], drop_first=True)

# Align columns with training features
X_dos = X_dos.reindex(columns=X.columns, fill_value=0)


In [ ]:
#vae

import numpy as np
import tensorflow as tf
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf

class Sampling(tf.keras.layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        z_log_var = tf.clip_by_value(z_log_var, -10.0, 10.0)
        epsilon = tf.random.normal(shape=tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon



In [ ]:
import tensorflow as tf
import joblib
import numpy as np

encoder = tf.keras.models.load_model(
    "/content/drive/MyDrive/IDS_VAE/encoder.keras",
    custom_objects={"Sampling": Sampling}
)

decoder = tf.keras.models.load_model(
    "/content/drive/MyDrive/IDS_VAE/decoder.keras"
)

In [ ]:
df_0=df_dos[df_dos['target']==0].sample(1,random_state=42)
df_1=df_dos[df_dos['target']==1].sample(1,random_state=420)
df_10=pd.concat([df_0,df_1])
df_10['target'] = df_10['Label'].apply(
    lambda x: 0 if x.lower() == 'benign' else 1
)

X_dos = df_10.drop(columns=['Label', 'target'])

# Same encoding as before
X_dos = pd.get_dummies(X_dos, columns=['Protocol'], drop_first=True)

# Align columns with training features
X_dos = X_dos.reindex(columns=X.columns, fill_value=0)
import joblib

scaler = joblib.load("/content/drive/MyDrive/IDS_VAE/scaler.pkl")
X_dos_scaled = scaler.transform(X_dos)
# reconstruction error function
def recon_error(encoder, decoder, data):
    z_mean, z_log_var, z = encoder(data)
    recon = decoder(z)
    return np.mean(np.square(data - recon.numpy()), axis=1)

# compute DoS reconstruction error
dos_error = recon_error(
    encoder,
    decoder,
    tf.convert_to_tensor(X_dos_scaled)
)

# apply threshold
dos_pred = (dos_error > threshold).astype(int)

# ground truth
y_true_dos = df_10['target'].values

# evaluation
print("Confusion Matrix (DoS):")
print(confusion_matrix(y_true_dos, dos_pred))

print("\nClassification Report (DoS):")
print(classification_report(y_true_dos, dos_pred))

Confusion Matrix (DoS):
[[1 0]
 [0 1]]

Classification Report (DoS):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2



In [ ]:
dos_error

array([0.2397078, 5.9375634])

In [ ]:
dos_pred

array([0, 1])

In [ ]:
#openset classifier
import tensorflow as tf
import pickle
import numpy as np
import pandas as pd
from scipy.stats import weibull_min

BASE_PATH = "/content/drive/MyDrive/IDS_OpenSet_Model"

# Load models
model = tf.keras.models.load_model(f"{BASE_PATH}/temporal_cnn.keras")
feature_extractor = tf.keras.models.load_model(f"{BASE_PATH}/feature_extractor.keras")

# Load label encoder
with open(f"{BASE_PATH}/label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

# Load OpenMax parameters
with open(f"{BASE_PATH}/openmax_params.pkl", "rb") as f:
    openmax_params = pickle.load(f)

class_means = openmax_params["class_means"]
class_thresholds = openmax_params["class_thresholds"]
train_distances = openmax_params["train_distances"]
weibull_models = openmax_params["weibull_models"]

# Load scaler
with open(f"{BASE_PATH}/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

print("All OpenSet components loaded successfully")


def openmax_predict(feature):
    distances = {}
    scores = {}

    for cls, mean in class_means.items():
        dist = np.linalg.norm(feature - mean)
        distances[cls] = dist

        w = weibull_models[cls]
        scores[cls] = 1.0 - weibull_min.cdf(dist, *w)

    best_cls = max(scores, key=scores.get)
    best_class_name = le.inverse_transform([best_cls])[0]

    # Primary rejection
    if distances[best_cls] > class_thresholds[best_cls]:
        return "NOVEL"

    # Secondary rejection
    if scores[best_cls] < 0.15:
        return "NOVEL"

    # Extra strictness for slowloris
    if best_class_name == "DoS slowloris":
        limit = np.percentile(train_distances[best_cls], 95)
        if distances[best_cls] > limit:
            return "NOVEL"

    return best_class_name
def preprocess_sequence(df, feature_cols, window=10):
    df = df.sort_values("pseudo_time")

    X = df[feature_cols].values

    # Create sliding windows
    X_seq = []
    for i in range(len(X) - window):
        X_seq.append(X[i:i+window])

    X_seq = np.array(X_seq)

    # Scale
    X_flat = X_seq.reshape(-1, X_seq.shape[-1])
    X_flat = scaler.transform(X_flat)
    X_seq = X_flat.reshape(X_seq.shape)

    return X_seq
known_df = pd.read_parquet("/content/drive/MyDrive/IDS_Datasets/OpenSet/cicids_known_attacks.parquet")
novel_df = pd.read_parquet("/content/drive/MyDrive/IDS_Datasets/OpenSet/cicids_novel_attacks.parquet")
# 10 known samples
known_10 = known_df.sample(10, random_state=42)

# 10 novel samples
novel_10 = novel_df.sample(10, random_state=420)

print("Known sample attack types:")
print(known_10["attack_type"].values)

print("\nNovel sample attack types:")
print(novel_10["attack_type"].values)
def preprocess_single(df):
    df = df.copy()

    # Add missing columns if any
    for col in FEATURE_COLS:
        if col not in df.columns:
            df[col] = 0

    # Keep correct order
    df = df[FEATURE_COLS]

    X = df.to_numpy()
    X_scaled = scaler.transform(X)

    return X_scaled
FEATURE_COLS = [
    c for c in known_df.columns
    if c not in ["Label", "attack_type", "pseudo_time", "source_file"]
]
# Known
X_known_scaled = preprocess_single(known_10)
WINDOW = 10

# Add fake time dimension (repeat each sample 10 times)
X_known_seq = np.repeat(X_known_scaled[:, np.newaxis, :], WINDOW, axis=1)

features_known = feature_extractor.predict(X_known_seq)


# Novel
X_novel_scaled = preprocess_single(novel_10)
X_novel_seq = np.repeat(X_novel_scaled[:, np.newaxis, :], WINDOW, axis=1)

features_novel = feature_extractor.predict(X_novel_seq)
print("Expected features:", len(FEATURE_COLS))
print("Scaler expects:", scaler.n_features_in_)



All OpenSet components loaded successfully
Known sample attack types:
['DoS Hulk' 'DoS Hulk' 'DoS Hulk' 'DoS Hulk' 'DoS Hulk' 'DoS Hulk' 'DDoS'
 'DoS Hulk' 'DoS Hulk' 'DoS Hulk']

Novel sample attack types:
['Bot' 'Bot' 'Bot' 'Bot' 'Bot' 'Bot' 'Bot' 'Bot' 'Bot' 'Bot']
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Expected features: 77
Scaler expects: 77


In [ ]:
for i, feat in enumerate(features_known):
    print("True:", known_10.iloc[i]["attack_type"])
    print("Pred:", openmax_predict(feat))
    print("-"*30)
print("----- Novel Sample Predictions -----")

for i, feat in enumerate(features_novel):
    pred = openmax_predict(feat)
    true = novel_10.iloc[i]["attack_type"]

    print(f"True: {true}  →  Predicted: {pred}")
    print("-"*40)

True: DoS Hulk
Pred: DoS Hulk
------------------------------
True: DoS Hulk
Pred: NOVEL
------------------------------
True: DoS Hulk
Pred: DoS Hulk
------------------------------
True: DoS Hulk
Pred: NOVEL
------------------------------
True: DoS Hulk
Pred: DoS Hulk
------------------------------
True: DoS Hulk
Pred: DoS Hulk
------------------------------
True: DDoS
Pred: DDoS
------------------------------
True: DoS Hulk
Pred: DoS Hulk
------------------------------
True: DoS Hulk
Pred: NOVEL
------------------------------
True: DoS Hulk
Pred: DoS Hulk
------------------------------
----- Novel Sample Predictions -----
True: Bot  →  Predicted: NOVEL
----------------------------------------
True: Bot  →  Predicted: NOVEL
----------------------------------------
True: Bot  →  Predicted: NOVEL
----------------------------------------
True: Bot  →  Predicted: NOVEL
----------------------------------------
True: Bot  →  Predicted: NOVEL
----------------------------------------
True: Bot 

In [ ]:
#novel_attack


import numpy as np
import pandas as pd
import tensorflow as tf
import joblib



BASE_PATH = "/content/drive/MyDrive/IDS_ContinualLearning"

model = tf.keras.models.load_model(f"{BASE_PATH}/continuallearning.keras")
scaler = joblib.load(f"{BASE_PATH}/scaler.pkl")
le = joblib.load(f"{BASE_PATH}/label_encoder.pkl")

print("✅ Model loaded successfully")
known_df = pd.read_parquet(
    "/content/drive/MyDrive/IDS_Datasets/OpenSet/cicids_known_attacks.parquet"
)

novel_df = pd.read_parquet(
    "/content/drive/MyDrive/IDS_Datasets/OpenSet/cicids_novel_attacks.parquet"
)


known_df = known_df[known_df["attack_type"] != "Heartbleed"]
novel_df = novel_df[novel_df["attack_type"] != "Heartbleed"]
FEATURE_COLS = [
    c for c in known_df.columns
    if c not in ["Label", "attack_type", "pseudo_time", "source_file"]
]

WINDOW = 10

def preprocess_and_sequence(df):
    df = df.copy()

    # Ensure all features exist
    for col in FEATURE_COLS:
        if col not in df.columns:
            df[col] = 0

    df = df[FEATURE_COLS]

    X = df.values
    X_scaled = scaler.transform(X)

    # Create temporal input (repeat row WINDOW times)
    X_seq = np.repeat(X_scaled[:, np.newaxis, :], WINDOW, axis=1)

    return X_seq
known_10 = known_df.sample(10, random_state=420)
novel_10 = novel_df.sample(10, random_state=4200)



print("\n--- KNOWN SAMPLES ---")

X_known_seq = preprocess_and_sequence(known_10)

pred_known = model.predict(X_known_seq)
pred_known_cls = np.argmax(pred_known, axis=1)

for i in range(10):
    true_label = known_10.iloc[i]["attack_type"]
    pred_label = le.inverse_transform([pred_known_cls[i]])[0]
    print(f"True: {true_label}  →  Predicted: {pred_label}")
print("\n--- NOVEL SAMPLES ---")

X_novel_seq = preprocess_and_sequence(novel_10)

pred_novel = model.predict(X_novel_seq)
pred_novel_cls = np.argmax(pred_novel, axis=1)

for i in range(10):
    true_label = novel_10.iloc[i]["attack_type"]
    pred_label = le.inverse_transform([pred_novel_cls[i]])[0]
    print(f"True: {true_label}  →  Predicted: {pred_label}")



✅ Model loaded successfully

--- KNOWN SAMPLES ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
True: DoS Hulk  →  Predicted: DoS Hulk
True: DDoS  →  Predicted: DDoS
True: DoS Hulk  →  Predicted: DoS Hulk
True: DoS Hulk  →  Predicted: DoS Hulk
True: DDoS  →  Predicted: DDoS
True: DoS Hulk  →  Predicted: DoS Hulk
True: SSH-Patator  →  Predicted: SSH-Patator
True: DoS GoldenEye  →  Predicted: DoS GoldenEye
True: DoS Slowhttptest  →  Predicted: DoS Slowhttptest
True: DDoS  →  Predicted: DDoS

--- NOVEL SAMPLES ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
True: Bot  →  Predicted: Bot
True: Bot  →  Predicted: Bot
True: Bot  →  Predicted: Bot
True: Bot  →  Predicted: Bot
True: Bot  →  Predicted: Bot
True: Bot  →  Predicted: Bot
True: Bot  →  Predicted: Bot
True: Bot  →  Predicted: Bot
True: Bot  →  Predicted: Bot
True: Bot  →  Predicted: Bot


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import tensorflow as tf
import joblib
import pickle
import time
from scipy.stats import weibull_min

# --- 1. PAGE CONFIGURATION ---
st.set_page_config(page_title="Adaptive IDS Pipeline", page_icon="🛡️", layout="wide")
st.title("🛡️ Cascading Adaptive IDS")
st.markdown("### Stage 1: VAE Gatekeeper ➜ Stage 2: Adaptive Classifier")

# --- 2. CUSTOM VAE LAYER ---
class Sampling(tf.keras.layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        z_log_var = tf.clip_by_value(z_log_var, -10.0, 10.0)
        epsilon = tf.random.normal(shape=tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

# --- 3. LOAD ALL MODELS (CACHED) ---
@st.cache_resource
def load_all_models():
    # --- VAE (Common) ---
    VAE_PATH = "/content/drive/MyDrive/IDS_VAE"
    vae_enc = tf.keras.models.load_model(f"{VAE_PATH}/encoder.keras", custom_objects={"Sampling": Sampling})
    vae_dec = tf.keras.models.load_model(f"{VAE_PATH}/decoder.keras")
    vae_scaler = joblib.load(f"{VAE_PATH}/scaler.pkl")

    # --- OPENSET (Before Training) ---
    OS_PATH = "/content/drive/MyDrive/IDS_OpenSet_Model"
    os_feat_ext = tf.keras.models.load_model(f"{OS_PATH}/feature_extractor.keras")
    with open(f"{OS_PATH}/label_encoder.pkl", "rb") as f: os_le = pickle.load(f)
    with open(f"{OS_PATH}/openmax_params.pkl", "rb") as f: os_params = pickle.load(f)
    with open(f"{OS_PATH}/scaler.pkl", "rb") as f: os_scaler = pickle.load(f)

    # --- CONTINUAL LEARNING (After Training) ---
    CL_PATH = "/content/drive/MyDrive/IDS_ContinualLearning"
    cl_model = tf.keras.models.load_model(f"{CL_PATH}/continuallearning.keras")
    cl_scaler = joblib.load(f"{CL_PATH}/scaler.pkl")
    cl_le = joblib.load(f"{CL_PATH}/label_encoder.pkl")

    # --- DATASETS ---
    known_df = pd.read_parquet("/content/drive/MyDrive/IDS_Datasets/OpenSet/cicids_known_attacks.parquet")
    novel_df = pd.read_parquet("/content/drive/MyDrive/IDS_Datasets/OpenSet/cicids_novel_attacks.parquet")

    return vae_enc, vae_dec, vae_scaler, os_feat_ext, os_scaler, os_le, os_params, cl_model, cl_scaler, cl_le, known_df, novel_df

# Run Loader
try:
    with st.spinner("Initializing Multi-Model Pipeline..."):
        vae_enc, vae_dec, vae_scaler, os_feat_ext, os_scaler, os_le, os_params, cl_model, cl_scaler, cl_le, known_df, novel_df = load_all_models()
    st.sidebar.success("✅ All Models Loaded")
except Exception as e:
    st.sidebar.error(f"Error loading components: {e}")
    st.stop()

# --- 4. GLOBAL FEATURE DEFINITION (THE FIX) ---
# This defines the 77 numerical features globally so 'analyze_traffic' can find them.
FEATURE_COLS = [c for c in known_df.columns if c not in ["Label", "attack_type", "pseudo_time", "source_file"]]
WINDOW = 10

# --- 5. SIDEBAR SETTINGS ---
st.sidebar.header("🕹️ Demo Controls")
model_mode = st.sidebar.radio(
    "Select Classifier State:",
    ("OpenSet (Before Training)", "Continual Learning (After Training)")
)

# --- 6. CORE LOGIC FUNCTIONS ---
def openmax_predict(feature_vector, os_params, os_le):
    distances = {}
    scores = {}
    for cls, mean in os_params["class_means"].items():
        dist = np.linalg.norm(feature_vector - mean)
        distances[cls] = dist
        w = os_params["weibull_models"][cls]
        scores[cls] = 1.0 - weibull_min.cdf(dist, *w)

    best_cls = max(scores, key=scores.get)
    best_name = os_le.inverse_transform([best_cls])[0]

    # Rejection logic
    is_novel = False
    if distances[best_cls] > os_params["class_thresholds"][best_cls]: is_novel = True
    elif scores[best_cls] < 0.15: is_novel = True

    final_label = "NOVEL" if is_novel else best_name
    return final_label, scores

# --- 7. ANALYSIS ENGINE ---
def analyze_traffic(sample_features, true_label):
    st.write("---")
    st.write(f"**Intercepting Flow...**")

    # STAGE 1: VAE
    st.markdown("#### 🟡 Stage 1: VAE Gatekeeper")
    sample_for_vae = sample_features.copy()
    if 'Protocol' in sample_for_vae.columns:
        sample_for_vae = pd.get_dummies(sample_for_vae, columns=['Protocol'])
        for p in ['Protocol_6', 'Protocol_17']:
            if p not in sample_for_vae.columns: sample_for_vae[p] = 0

    vae_cols = getattr(vae_scaler, 'feature_names_in_', None)
    sample_for_vae = sample_for_vae.reindex(columns=vae_cols if vae_cols is not None else sample_for_vae.columns, fill_value=0)

    v_scaled = vae_scaler.transform(sample_for_vae.values)
    z_mean, z_log_var, z = vae_enc(tf.convert_to_tensor(v_scaled, dtype=tf.float32))
    recon = vae_dec(z)
    err = np.mean(np.square(v_scaled - recon.numpy()), axis=1)[0]

    st.metric("Reconstruction Error", f"{err:.4f}", "Normal: 0.2 - 0.8")

    if 0.200 <= err <= 0.809:
        st.success("🟢 Benign Traffic. Dropped.")
        return

    st.warning("🔴 Anomaly Found. Forwarding...")

    # STAGE 2: CLASSIFIER
    st.markdown(f"#### 🔵 Stage 2: {model_mode}")
    time.sleep(0.5)

    # Use global FEATURE_COLS to align
    s_aligned = sample_features.reindex(columns=FEATURE_COLS, fill_value=0)

    if model_mode == "OpenSet (Before Training)":
        s_scaled = os_scaler.transform(s_aligned.values)
        X_seq = np.repeat(s_scaled[:, np.newaxis, :], WINDOW, axis=1)
        feats = os_feat_ext.predict(X_seq, verbose=0)[0]
        pred_label, scores = openmax_predict(feats, os_params, os_le)

        chart_scores = {os_le.inverse_transform([k])[0]: v for k, v in scores.items()}
        st.bar_chart(pd.DataFrame({"Score": chart_scores}))
    else:
        s_scaled = cl_scaler.transform(s_aligned.values)
        X_seq = np.repeat(s_scaled[:, np.newaxis, :], WINDOW, axis=1)
        logits = cl_model.predict(X_seq, verbose=0)[0]
        probs = tf.nn.softmax(logits).numpy()
        pred_label = cl_le.inverse_transform([np.argmax(probs)])[0]
        st.bar_chart(pd.DataFrame({"Score": probs}, index=cl_le.classes_))

    if pred_label == "NOVEL" or (pred_label != true_label and true_label != 'Benign'):
        st.error(f"🚨 ALERT: {pred_label} (Ground Truth: {true_label})")
    else:
        st.success(f"🛑 BLOCKED: {pred_label} (Ground Truth: {true_label})")

# --- 8. DATABASE SETUP ---
@st.cache_data
def create_demo_database():
    demo_known = known_df.sample(5, random_state=42).copy()
    demo_known['Threat_Category'] = 'Known Attack'

    demo_novel = novel_df.sample(5, random_state=420).copy()
    demo_novel['Threat_Category'] = 'Novel Attack (Zero-Day)'

    demo_benign = pd.DataFrame(np.zeros((5, len(known_df.columns))), columns=known_df.columns)
    demo_benign['attack_type'] = 'Benign'
    demo_benign['Threat_Category'] = 'Normal Traffic'

    db = pd.concat([demo_benign, demo_known, demo_novel]).reset_index(drop=True)
    db['Flow_ID'] = [f"FLOW_100{i}" for i in range(len(db))]
    db = db.sample(frac=1, random_state=99).reset_index(drop=True)
    return db

demo_db = create_demo_database()

# --- 8. UI CONTROLS ---
st.subheader("🗄️ Network Traffic Database (Demo)")
display_df = demo_db[['Flow_ID', 'Threat_Category', 'attack_type']]
st.dataframe(display_df, use_container_width=True, hide_index=True)

st.write("---")
st.subheader("🔬 Intercept & Analyze")

selected_flow_id = st.selectbox("Select a Flow ID to intercept:", demo_db['Flow_ID'])

if st.button("Run IDS Pipeline on Selected Flow"):
    selected_row = demo_db[demo_db['Flow_ID'] == selected_flow_id].iloc[0]
    true_label = selected_row['attack_type']
    sample_features = selected_row[FEATURE_COLS].to_frame().T

    analyze_traffic(sample_features, true_label)

Writing app.py


In [ ]:
# 1. Kill existing Streamlit and Tunnel processes
!pkill -f streamlit
!pkill -f cloudflared

# 2. Start Streamlit in the background
!nohup streamlit run app.py --server.enableCORS=false --server.enableXsrfProtection=false &

# 3. Download and start Cloudflare Tunnel (No account required)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!nohup ./cloudflared-linux-amd64 tunnel --url http://localhost:8501 > cloudflare.log 2>&1 &

# 4. Extract and print the public link
import time
time.sleep(3)
!grep -o 'https://[-0-9a-z]*\.trycloudflare.com' cloudflare.log

nohup: appending output to 'nohup.out'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1. Kill any stuck tunnels
!pkill -f cloudflared

# 2. Download Cloudflare (if you haven't already)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

print("⏳ Starting Tunnel... Look for the link ending in .trycloudflare.com below!")
print("----------------------------------------------------------------------")

# 3. Run the tunnel in the FOREGROUND so it prints the link right here
!./cloudflared-linux-amd64 tunnel --url http://localhost:8501

⏳ Starting Tunnel... Look for the link ending in .trycloudflare.com below!
----------------------------------------------------------------------
2026-03-27T03:39:21Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-03-27T03:39:21Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-27T03:39:24Z INF +--------------------------------------------------------------------------------------------+
2026-03-27T03:39:24Z INF |  Your quick Tunnel has been 

In [ ]:
import time

# 1. Kill any hung background processes
!pkill -f streamlit
!pkill -f cloudflared

print("🔄 Restarting Streamlit on strict IPv4...")
# 2. Start Streamlit (forcing it to 0.0.0.0)
!nohup streamlit run app.py --server.address=0.0.0.0 --server.port=8501 --server.enableCORS=false --server.enableXsrfProtection=false > streamlit.log 2>&1 &

# Wait 5 seconds to ensure Streamlit is fully awake before connecting the tunnel
time.sleep(5)

print("🔗 Attaching Cloudflare Tunnel...")
# 3. Start Cloudflare (pointing explicitly to 127.0.0.1 instead of localhost)
!nohup ./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:8501 > cloudflare.log 2>&1 &

# Wait 5 seconds for Cloudflare to assign the URL
time.sleep(5)

print("\n✅ READY! Click your new link below:")
print("--------------------------------------------------")
# 4. Print the new link
!grep -o 'https://[-0-9a-z]*\.trycloudflare.com' cloudflare.log

🔄 Restarting Streamlit on strict IPv4...
🔗 Attaching Cloudflare Tunnel...

✅ READY! Click your new link below:
--------------------------------------------------


In [ ]:
!tail -n 50 streamlit.log




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.190.136.111:8501



In [ ]:
# 1. Install Streamlit and all required data processing libraries
!pip install -q streamlit tensorflow scikit-learn joblib pandas numpy scipy pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 75.3 MB/s eta 0:00:00


In [ ]:
import time
import os

# 1. Kill any hung background processes
!pkill -f streamlit
!pkill -f cloudflared

print("🚀 Starting Streamlit (Loading models - this takes a moment)...")

# 2. Use 'python -m streamlit' to avoid PATH issues
!nohup python -m streamlit run app.py --server.address=0.0.0.0 --server.port=8501 --server.enableCORS=false --server.enableXsrfProtection=false > streamlit.log 2>&1 &

# 3. Wait for Streamlit to actually be ready (don't just guess with sleep)
ready = False
for i in range(30):
    if os.path.exists("streamlit.log"):
        with open("streamlit.log", "r") as f:
            if "Network URL:" in f.read():
                print("✅ Streamlit is AWAKE and listening on port 8501!")
                ready = True
                break
    time.sleep(2)
    if i % 5 == 0: print(f"Still loading... ({i*2}s)")

if not ready:
    print("❌ Streamlit failed to start. Check the logs with: !tail -n 20 streamlit.log")
else:
    print("🔗 Attaching Cloudflare Tunnel...")
    # 4. Start Cloudflare
    !nohup ./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:8501 > cloudflare.log 2>&1 &

    # 5. Find and print the link
    time.sleep(8)
    print("\n✅ READY! Your public URL is:")
    print("--------------------------------------------------")
    !grep -o 'https://[-0-9a-z]*\.trycloudflare.com' cloudflare.log

🚀 Starting Streamlit (Loading models - this takes a moment)...
Still loading... (0s)
✅ Streamlit is AWAKE and listening on port 8501!
🔗 Attaching Cloudflare Tunnel...

✅ READY! Your public URL is:
--------------------------------------------------


In [ ]:
!tail -n 20 streamlit.log

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py


In [ ]:
!ls -l app.py

ls: cannot access 'app.py': No such file or directory


In [ ]:
# 1. Download the Cloudflare binary
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64

# 2. Give it 'Execute' permissions (this is why the previous command failed)
!chmod +x cloudflared-linux-amd64

print("✅ Cloudflare bridge is ready! Now run the tunnel cell again.")

✅ Cloudflare bridge is ready! Now run the tunnel cell again.


In [ ]:
import time
import re
import os

# 1. Start the tunnel
if os.path.exists("cloudflared-linux-amd64"):
    print("🛰️ Connecting Cloudflare Tunnel...")
    !nohup ./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:8501 > cloudflare.log 2>&1 &
else:
    print("❌ ERROR: Binary still missing. Run the download cell above first!")

# 2. Wait and search for the link
print("⏳ Waiting for public URL...")
time.sleep(12)

if os.path.exists("cloudflare.log"):
    with open("cloudflare.log", "r") as f:
        log_data = f.read()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_data)
        if match:
            public_url = match.group(0)
            print("\n" + "═"*50)
            print(f"🚀 PROJECT IS LIVE!")
            print(f"🔗 URL: {public_url}")
            print("═"*50)
        else:
            print("❌ Link not found yet. Try running this cell again in 10 seconds.")
            !tail -n 5 cloudflare.log

🛰️ Connecting Cloudflare Tunnel...
⏳ Waiting for public URL...

══════════════════════════════════════════════════
🚀 PROJECT IS LIVE!
🔗 URL: https://machine-contributing-numerical-changelog.trycloudflare.com
══════════════════════════════════════════════════
